# Documentacao do `index.js`

Este notebook foi atualizado para refletir exatamente o fluxo implementado hoje em `index.js`.
A proposta e estudar o algoritmo, nao apenas mostrar comandos soltos do TensorFlow.js.

## 1. Visao geral do algoritmo

O arquivo `index.js` implementa um classificador supervisionado com 3 classes.

Fluxo geral:

1. configurar o backend do TensorFlow.js;
2. criar os dados de treino;
3. transformar esses dados em tensores;
4. montar a rede neural;
5. treinar com `fit(...)`;
6. prever a classe de uma nova entrada.

In [ ]:
import * as tf from '@tensorflow/tfjs';

await tf.setBackend('cpu');
await tf.ready();

console.log('Backend atual:', tf.getBackend());

## 2. Configuracao inicial

O codigo usa `@tensorflow/tfjs` com backend `cpu`:

```js
import * as tf from '@tensorflow/tfjs';
await tf.setBackend('cpu');
await tf.ready();
```

Essa escolha evita o problema de compatibilidade que ocorreu com `@tensorflow/tfjs-node` no ambiente atual.

In [ ]:
const tensorPessoas = [
  [0.33, 1, 0, 0, 1, 0],
  [0.00, 0, 1, 0, 0, 1],
  [1.00, 0, 0, 1, 1, 0]
];

const tensorCategoriaPessoa = [
  [1, 0, 0],
  [0, 1, 0],
  [0, 0, 1]
];

const inputTensorX = tf.tensor2d(tensorPessoas);
const outTensorY = tf.tensor2d(tensorCategoriaPessoa);

console.log('inputTensorX.shape =', inputTensorX.shape);
console.log('outTensorY.shape =', outTensorY.shape);

## 3. Dados de entrada e saida

O tensor de entrada tem shape `[3, 6]`:

- 3 exemplos;
- 6 atributos por exemplo.

O tensor de saida tem shape `[3, 3]`:

- 3 exemplos;
- 3 classes.

A saida esta em one-hot encoding. Por isso a camada final usa `softmax` e a perda usa `categoricalCrossentropy`.

In [ ]:
async function trainModel(input, outPut) {
  const model = tf.sequential();

  model.add(tf.layers.dense({
    inputShape: [6],
    units: 16,
    activation: 'elu'
  }));

  model.add(tf.layers.dense({
    units: 3,
    activation: 'softmax'
  }));

  model.compile({
    optimizer: 'adam',
    loss: 'categoricalCrossentropy',
    metrics: ['accuracy']
  });

  await model.fit(input, outPut, {
    verbose: 0,
    epochs: 100,
    shuffle: true,
    callbacks: {
      onEpochEnd: async (epoch, log) => {
        console.log(`epoch: ${epoch + 1}: loss = ${log.loss}`);
      }
    }
  });

  return model;
}

## 4. Estrutura da funcao `trainModel`

A funcao `trainModel` faz quatro coisas:

1. cria um modelo sequencial com `tf.sequential()`;
2. adiciona duas camadas densas;
3. compila o modelo;
4. treina o modelo com `fit(...)` e depois o retorna.

O `await` dentro da funcao garante que o retorno acontece apenas depois do treinamento terminar.

## 5. Arquitetura da rede neural

A rede possui duas camadas:

- primeira camada: `Dense(16, activation='elu')`;
- segunda camada: `Dense(3, activation='softmax')`.

Interpretacao:

- `inputShape: [6]` porque cada exemplo possui 6 valores;
- `units: 16` cria 16 neuronios na camada oculta;
- `elu` ajuda a aprender relacoes nao lineares;
- `units: 3` na saida representa as 3 classes do problema;
- `softmax` converte a saida final em probabilidades.

## 6. Compilacao e treino

O modelo e compilado com:

- `optimizer: 'adam'`;
- `loss: 'categoricalCrossentropy'`;
- `metrics: ['accuracy']`.

Durante o treino, em cada epoca, o TensorFlow.js:

1. calcula previsoes para `inputTensorX`;
2. compara com `outTensorY`;
3. calcula o erro;
4. ajusta os pesos da rede.

O callback `onEpochEnd` imprime o valor de `loss` para acompanhar a convergencia.

In [ ]:
const model = await trainModel(inputTensorX, outTensorY);

const novaPessoa = tf.tensor2d([
  [0.20, 1, 0, 0, 1, 0]
]);

const prediction = model.predict(novaPessoa);
prediction.print();

## 7. Predicao no codigo atual

Depois do treino, o codigo cria um novo tensor com uma pessoa e executa:

```js
const prediction = model.predict(novaPessoa)
prediction.print()
```

O retorno e um tensor com 3 probabilidades. Cada posicao corresponde a uma das 3 classes aprendidas pelo modelo.

## 8. Como interpretar a saida

Se a previsao for algo como:

```text
[[0.68, 0.10, 0.21]]
```

isso significa:

- classe 0: 68% de confianca;
- classe 1: 10% de confianca;
- classe 2: 21% de confianca.

A classe escolhida pelo modelo e a que tiver o maior valor.

## 9. Resumo final

O `index.js` atual implementa este pipeline:

1. inicializar o TensorFlow.js com backend `cpu`;
2. criar tensores 2D de entrada e saida;
3. montar uma rede sequencial com uma camada oculta e uma camada de saida;
4. compilar com `adam` e `categoricalCrossentropy`;
5. treinar por 100 epocas;
6. prever a categoria de uma nova pessoa.

Esse notebook agora descreve diretamente esse fluxo, com os mesmos nomes de variaveis e a mesma ordem usada no arquivo `index.js`.